In [17]:
import glob
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data_path = r"C:\Users\sanst_5a1e4f1\Downloads\archive (2)"

if os.path.isdir(data_path):
  # Grab all .xlsx files (excluding index sheets if you only want individual stocks)
  xlsx_files = sorted(glob.glob(os.path.join(data_path, "**", "*.xlsx"), recursive=True))
  stock_files = [f for f in xlsx_files if not os.path.basename(f).startswith("Index_")]
  files_to_load = stock_files if stock_files else xlsx_files

  print(f"Found {len(xlsx_files)} .xlsx files; loading {len(files_to_load)} files...")

  frames = []
  for f in files_to_load:
    temp = pd.read_excel(f)
    if "Symbol" not in temp.columns and "Ticker" not in temp.columns:
      temp["Symbol"] = os.path.splitext(os.path.basename(f))[0]
    frames.append(temp)

  df = pd.concat(frames, ignore_index=True)
else:
  df = pd.read_excel(data_path) if data_path.endswith(".xlsx") else pd.read_csv(data_path)

df.columns = [str(c).strip() for c in df.columns]
print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

# --- 1. INCOMPLETE VALUES ---
missing_counts = df.isna().sum()
print("\n--- Missing Values ---")
print(missing_counts[missing_counts > 0] if (missing_counts > 0).any() else "No NaNs found")

# --- 2. DUPLICATE VALUES ---
dup_rows = df.duplicated().sum()
subset_cols = [c for c in ["Date", "Symbol", "Ticker"] if c in df.columns]
dup_keys = df.duplicated(subset=subset_cols).sum() if len(subset_cols) >= 2 else 0
print(f"\nExact duplicate rows: {dup_rows}")
if len(subset_cols) >= 2:
  print(f"Duplicate {subset_cols} entries: {dup_keys}")

# --- 3. INCORRECT VALUES ---
col_map = {c.lower(): c for c in df.columns}
open_c = col_map.get("open")
high_c = col_map.get("high")
low_c = col_map.get("low")
close_c = col_map.get("close") or col_map.get("last") or col_map.get("ltp")
vol_c = col_map.get("volume") or col_map.get("tottrdqnty") or col_map.get("shares traded")

price_cols = [c for c in [open_c, high_c, low_c, close_c] if c is not None]
bad_price_count = int((df[price_cols] <= 0).any(axis=1).sum()) if price_cols else 0
bad_hl_count = int((df[high_c] < df[low_c]).sum()) if (high_c and low_c) else 0
neg_vol_count = int((df[vol_c] < 0).sum()) if vol_c else 0

print(f"\nNon-positive (<= 0) price rows: {bad_price_count}")
print(f"Invalid rows where High < Low: {bad_hl_count}")
print(f"Negative volume rows: {neg_vol_count}")

Found 103 .xlsx files; loading 100 files...
Shape: (244901, 15)
Columns: ['Date', 'Symbol', 'Series', 'Prev Close', 'Open', 'High', 'Low', 'Last', 'Close', 'VWAP', 'Volume', 'Turnover', 'Trades', 'Deliverable Volume', '%Deliverble']


,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble
0,2010-01-08,ABBOTINDIA,EQ,783.75,755.0,807.0,755.0,807.0,807.0,784.12,51,3.998995e+09,NaN,50,0.9804
1,2010-01-11,ABBOTINDIA,EQ,807.00,825.0,825.0,790.0,804.8,804.8,793.90,897,7.121265e+10,NaN,697,0.7770
2,2010-01-12,ABBOTINDIA,EQ,804.80,800.0,805.0,790.0,790.0,790.0,799.36,426,3.405260e+10,NaN,401,0.9413
3,2010-01-13,ABBOTINDIA,EQ,790.00,787.0,787.0,720.0,767.0,767.0,758.53,359,2.723129e+10,NaN,164,0.4568
4,2010-01-14,ABBOTINDIA,EQ,767.00,769.0,798.0,769.0,785.1,785.1,791.73,317,2.509785e+10,NaN,122,0.3849



--- Missing Values ---
Trades    30684
dtype: int64

Exact duplicate rows: 264
Duplicate ['Date', 'Symbol'] entries: 264

Non-positive (<= 0) price rows: 0
Invalid rows where High < Low: 0
Negative volume rows: 0


In [18]:
df_clean = df.copy()

# 1. Remove duplicate rows
df_clean = df_clean.drop_duplicates()
if len(subset_cols) >= 2:
  df_clean = df_clean.drop_duplicates(subset=subset_cols, keep="first")

# 2. Fix High < Low by swapping when both exist
if high_c and low_c:
  swap_mask = df_clean[high_c] < df_clean[low_c]
  df_clean.loc[swap_mask, [high_c, low_c]] = df_clean.loc[swap_mask, [low_c, high_c]].values

# 3. Replace non-positive prices / negative volume with NaN so we can clean/impute properly
for c in price_cols:
  df_clean.loc[df_clean[c] <= 0, c] = np.nan
if vol_c:
  df_clean.loc[df_clean[vol_c] < 0, vol_c] = np.nan

# 4. Sort by date/symbol if Date exists, then interpolate / fill missing numeric values
date_c = col_map.get("date")
if date_c:
  df_clean[date_c] = pd.to_datetime(df_clean[date_c], errors="coerce")
  sort_cols = [c for c in ["Symbol", "Ticker", date_c] if c in df_clean.columns]
  df_clean = df_clean.sort_values(sort_cols).dropna(subset=[date_c])

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
for c in numeric_cols:
  df_clean[c] = df_clean[c].interpolate(method="linear", limit_direction="both")
  df_clean[c] = df_clean[c].fillna(df_clean[c].median())

# Drop any remaining rows where target is still NaN
target_col = close_c or numeric_cols[-1]
df_clean = df_clean.dropna(subset=[target_col]).copy()

# Optional helpful engineered features (daily range & return) if Open/High/Low/Close exist
if open_c and high_c and low_c and close_c:
  df_clean["intraday_range"] = df_clean[high_c] - df_clean[low_c]
  df_clean["daily_change"] = df_clean[close_c] - df_clean[open_c]

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c != target_col]

X = df_clean[feature_cols]
y = df_clean[target_col]

# 5. Train/test split BEFORE fitting scaler (prevents data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Cleaned shape:", df_clean.shape)
print("Features used:", feature_cols)
print("Target column:", target_col)

Cleaned shape: (244637, 17)
Features used: ['Prev Close', 'Open', 'High', 'Low', 'Last', 'VWAP', 'Volume', 'Turnover', 'Trades', 'Deliverable Volume', '%Deliverble', 'intraday_range', 'daily_change']
Target column: Close


In [19]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

clf_df = df_clean.copy()
clf_df["target_up"] = (clf_df["Close"] > clf_df["Open"]).astype(int)

exclude_cols = {"Close", "target_up", "daily_change", "daily_return_pct"}
feature_cols = [
    c
    for c in clf_df.select_dtypes(include=[np.number]).columns
    if c not in exclude_cols
]

X = clf_df[feature_cols]
y = clf_df["target_up"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = []
print("Features:", feature_cols)
print("Train:", len(X_train), "| Test:", len(X_test))

Features: ['Prev Close', 'Open', 'High', 'Low', 'Last', 'VWAP', 'Volume', 'Turnover', 'Trades', 'Deliverable Volume', '%Deliverble', 'intraday_range']
Train: 195709 | Test: 48928


In [20]:
#randomforest
rf_clf = RandomForestClassifier(
    n_estimators=100, max_depth=15, n_jobs=-1, random_state=42
)
rf_clf.fit(X_train_scaled, y_train)
rf_pred = rf_clf.predict(X_test_scaled)

rf_acc = accuracy_score(y_test, rf_pred)
rf_prec = precision_score(y_test, rf_pred, zero_division=0)
rf_rec = recall_score(y_test, rf_pred, zero_division=0)

results.append({
    "Model": "Random Forest",
    "Accuracy": round(rf_acc, 4),
    "Precision": round(rf_prec, 4),
    "Recall": round(rf_rec, 4),
})
print(
    f"Random Forest -> Acc: {rf_acc:.4f} | Prec: {rf_prec:.4f} | Rec:"
    f" {rf_rec:.4f}"
)

Random Forest -> Acc: 0.6502 | Prec: 0.8616 | Rec: 0.2871


In [21]:
#logisticregression
lr_clf = LogisticRegression(max_iter=1000, random_state=42)
lr_clf.fit(X_train_scaled, y_train)
lr_pred = lr_clf.predict(X_test_scaled)

lr_acc = accuracy_score(y_test, lr_pred)
lr_prec = precision_score(y_test, lr_pred, zero_division=0)
lr_rec = recall_score(y_test, lr_pred, zero_division=0)

results.append({
    "Model": "Logistic Regression",
    "Accuracy": round(lr_acc, 4),
    "Precision": round(lr_prec, 4),
    "Recall": round(lr_rec, 4),
})
print(
    f"Logistic Regression -> Acc: {lr_acc:.4f} | Prec: {lr_prec:.4f} | Rec:"
    f" {lr_rec:.4f}"
)

Logistic Regression -> Acc: 0.8119 | Prec: 0.9707 | Rec: 0.6103


In [22]:
#gradientboost
gb_clf = HistGradientBoostingClassifier(max_iter=200, random_state=42)
gb_clf.fit(X_train_scaled, y_train)
gb_pred = gb_clf.predict(X_test_scaled)

gb_acc = accuracy_score(y_test, gb_pred)
gb_prec = precision_score(y_test, gb_pred, zero_division=0)
gb_rec = recall_score(y_test, gb_pred, zero_division=0)

results.append({
    "Model": "Gradient Boosting",
    "Accuracy": round(gb_acc, 4),
    "Precision": round(gb_prec, 4),
    "Recall": round(gb_rec, 4),
})
print(
    f"Gradient Boosting -> Acc: {gb_acc:.4f} | Prec: {gb_prec:.4f} | Rec:"
    f" {gb_rec:.4f}"
)

Gradient Boosting -> Acc: 0.8133 | Prec: 0.9167 | Rec: 0.6543


In [23]:
#decisiontreeclassifier
dt_clf = DecisionTreeClassifier(max_depth=15, random_state=42)
dt_clf.fit(X_train_scaled, y_train)
dt_pred = dt_clf.predict(X_test_scaled)

dt_acc = accuracy_score(y_test, dt_pred)
dt_prec = precision_score(y_test, dt_pred, zero_division=0)
dt_rec = recall_score(y_test, dt_pred, zero_division=0)

results.append({
    "Model": "Decision Tree",
    "Accuracy": round(dt_acc, 4),
    "Precision": round(dt_prec, 4),
    "Recall": round(dt_rec, 4),
})
print(
    f"Decision Tree -> Acc: {dt_acc:.4f} | Prec: {dt_prec:.4f} | Rec:"
    f" {dt_rec:.4f}"
)

Decision Tree -> Acc: 0.5684 | Prec: 0.6006 | Rec: 0.1895


In [24]:
comparison_df = pd.DataFrame(results).sort_values(
    by="Accuracy", ascending=False
)
display(comparison_df)

,Model,Accuracy,Precision,Recall
2,Gradient Boosting,0.8133,0.9167,0.6543
1,Logistic Regression,0.8119,0.9707,0.6103
0,Random Forest,0.6502,0.8616,0.2871
3,Decision Tree,0.5684,0.6006,0.1895
